# 09 — Model Registry & Deployment

**Objective**: Register the best model in the MLFlow model registry, save preprocessing
artifacts needed for serving, build a FastAPI scoring endpoint, and create a Streamlit
dashboard for interactive credit-risk assessment.

## Deployment Stack

| Component | Tool | Description |
|-----------|------|-------------|
| Model registry | MLFlow | Version control, stage management |
| Model serving | FastAPI | REST API for real-time predictions |
| Dashboard | Streamlit | Interactive UI for analysts |
| Preprocessing | scikit-learn OrdinalEncoder | Saved as pickle artifact |

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import json
import pickle
import subprocess
import time
from pathlib import Path
import warnings

import xgboost as xgb
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer

import mlflow
import mlflow.xgboost
from mlflow.models.signature import infer_signature

warnings.filterwarnings("ignore")

# Project paths
PROJECT_ROOT = Path("..").resolve()
FEATURES_DIR = PROJECT_ROOT / "data" / "features"    # Gold layer
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# MLFlow setup
MLFLOW_URI = f"sqlite:///{PROJECT_ROOT}/mlruns/mlflow.db"
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment("credit-risk-default-prediction")

RANDOM_STATE = 42

print(f"Project root:   {PROJECT_ROOT}")
print(f"Features dir:   {FEATURES_DIR}")
print(f"Models dir:     {MODELS_DIR}")
print(f"MLFlow URI:     {MLFLOW_URI}")

/home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project root:   /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform
Features dir:   /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/data/features
Models dir:     /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/models
MLFlow URI:     sqlite:////home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/mlruns/mlflow.db


## 2. Register Model in MLFlow

In [2]:
# Load gold layer features for signature inference
df = pd.read_parquet(FEATURES_DIR / "train_features.parquet")

y = df["TARGET"]
X = df.drop(columns=["SK_ID_CURR", "TARGET"])

numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Gold layer shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"  Numeric features:     {len(numeric_cols):>4}")
print(f"  Categorical features: {len(categorical_cols):>4}")
print(f"  Total features:       {X.shape[1]:>4}")

Gold layer shape: 307,511 rows x 227 columns
  Numeric features:      209
  Categorical features:   16
  Total features:        225


In [3]:
# Load the best XGBoost model
model_path = MODELS_DIR / "best_xgb.json"
booster = xgb.Booster()
booster.load_model(str(model_path))

print(f"Model loaded from: {model_path}")
print(f"  Number of trees: {booster.num_boosted_rounds()}")

Model loaded from: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/models/best_xgb.json
  Number of trees: 957


In [4]:
# Prepare data for signature inference
# Apply the same preprocessing as training: impute + ordinal-encode categoricals
X_sig = X.copy()

# Impute categoricals with most frequent then ordinal-encode
cat_imputer = SimpleImputer(strategy="most_frequent")
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)

if categorical_cols:
    X_sig[categorical_cols] = cat_imputer.fit_transform(X_sig[categorical_cols])
    X_sig[categorical_cols] = encoder.fit_transform(X_sig[categorical_cols])

# Get predictions for signature
sample_input = X_sig.head(5)
sample_preds = booster.predict(xgb.DMatrix(sample_input))

# Infer MLFlow signature
signature = infer_signature(sample_input, sample_preds)
print("Model signature inferred:")
print(signature)

Model signature inferred:
inputs: 
  ['NAME_CONTRACT_TYPE': double (required), 'CODE_GENDER': double (required), 'FLAG_OWN_CAR': double (required), 'FLAG_OWN_REALTY': double (required), 'CNT_CHILDREN': integer (required), 'AMT_INCOME_TOTAL': float (required), 'AMT_CREDIT': float (required), 'AMT_ANNUITY': float (required), 'AMT_GOODS_PRICE': float (required), 'NAME_TYPE_SUITE': double (required), 'NAME_INCOME_TYPE': double (required), 'NAME_EDUCATION_TYPE': double (required), 'NAME_FAMILY_STATUS': double (required), 'NAME_HOUSING_TYPE': double (required), 'REGION_POPULATION_RELATIVE': float (required), 'DAYS_BIRTH': integer (required), 'DAYS_EMPLOYED': double (required), 'DAYS_REGISTRATION': float (required), 'DAYS_ID_PUBLISH': integer (required), 'OWN_CAR_AGE': float (optional), 'FLAG_MOBIL': integer (required), 'FLAG_EMP_PHONE': integer (required), 'FLAG_WORK_PHONE': integer (required), 'FLAG_CONT_MOBILE': integer (required), 'FLAG_PHONE': integer (required), 'FLAG_EMAIL': integer (r

In [5]:
# Log model to MLFlow and register it
MODEL_NAME = "credit-risk-xgb"

with mlflow.start_run(run_name="register_best_xgb") as run:
    # Log the XGBoost model with signature
    mlflow.xgboost.log_model(
        xgb_model=booster,
        artifact_path="model",
        signature=signature,
        registered_model_name=MODEL_NAME,
    )

    # Log model metadata as params
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_features", len(X.columns))
    mlflow.log_param("n_trees", booster.num_boosted_rounds())
    mlflow.log_param("source_file", "best_xgb.json")

    run_id = run.info.run_id
    print(f"Model logged to MLFlow")
    print(f"  Run ID:          {run_id}")
    print(f"  Model URI:       runs:/{run_id}/model")
    print(f"  Registered as:   {MODEL_NAME}")

2026/03/25 17:25:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'credit-risk-xgb' already exists. Creating a new version of this model...
Created version '6' of model 'credit-risk-xgb'.


Model logged to MLFlow
  Run ID:          758cb384133c43e4bd0949875f79bf59
  Model URI:       runs:/758cb384133c43e4bd0949875f79bf59/model
  Registered as:   credit-risk-xgb


In [6]:
# Transition the latest version to Production
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get the latest version
latest_versions = client.get_latest_versions(MODEL_NAME)
latest_version = latest_versions[0]

print(f"Latest model version: {latest_version.version}")
print(f"  Current stage: {latest_version.current_stage}")

# Try alias-based approach (MLFlow >= 2.x), fall back to stage transition
try:
    client.set_registered_model_alias(MODEL_NAME, "production", latest_version.version)
    print(f"  Set alias 'production' -> version {latest_version.version}")
except Exception:
    # Fall back to stage-based transition (older MLFlow)
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version.version,
        stage="Production",
        archive_existing_versions=True,
    )
    print(f"  Transitioned to stage: Production")

# Print registered model info
registered_model = client.get_registered_model(MODEL_NAME)
print(f"\nRegistered Model: {registered_model.name}")
print(f"  Description: {registered_model.description or '(none)'}")
for mv in client.get_latest_versions(MODEL_NAME):
    print(f"  Version {mv.version}: stage={mv.current_stage}, run_id={mv.run_id[:8]}...")

Latest model version: 6
  Current stage: None
  Set alias 'production' -> version 6

Registered Model: credit-risk-xgb
  Description: (none)
  Version 6: stage=None, run_id=758cb384...


## 3. Save Preprocessing Artifacts

In [7]:
# Save the fitted OrdinalEncoder
preprocessor_path = MODELS_DIR / "preprocessor.pkl"
with open(preprocessor_path, "wb") as f:
    pickle.dump(encoder, f)
print(f"Preprocessor saved: {preprocessor_path}")
print(f"  Type:       {type(encoder).__name__}")
print(f"  Categories: {len(encoder.categories_)} columns encoded")

Preprocessor saved: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/models/preprocessor.pkl
  Type:       OrdinalEncoder
  Categories: 16 columns encoded


In [8]:
# Save feature column list (ordered)
feature_cols_path = MODELS_DIR / "feature_columns.json"
feature_columns = X.columns.tolist()
with open(feature_cols_path, "w") as f:
    json.dump(feature_columns, f, indent=2)
print(f"Feature columns saved: {feature_cols_path}")
print(f"  Total features: {len(feature_columns)}")
print(f"  First 10: {feature_columns[:10]}")

Feature columns saved: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/models/feature_columns.json
  Total features: 225
  First 10: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE']


In [9]:
# Save categorical column names
cat_cols_path = MODELS_DIR / "categorical_columns.json"
with open(cat_cols_path, "w") as f:
    json.dump(categorical_cols, f, indent=2)
print(f"Categorical columns saved: {cat_cols_path}")
print(f"  Total: {len(categorical_cols)}")
print(f"  Columns: {categorical_cols}")

Categorical columns saved: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/models/categorical_columns.json
  Total: 16
  Columns: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']


In [10]:
# Save model metrics for the dashboard
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score

# Re-create the validation split to compute metrics
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y,
)

# Preprocess validation data the same way
X_val_proc = X_val.copy()
if categorical_cols:
    X_val_proc[categorical_cols] = cat_imputer.transform(X_val_proc[categorical_cols])
    X_val_proc[categorical_cols] = encoder.transform(X_val_proc[categorical_cols])

y_proba = booster.predict(xgb.DMatrix(X_val_proc))
y_pred = (y_proba >= 0.5).astype(int)

metrics = {
    "auc_roc": round(roc_auc_score(y_val, y_proba), 4),
    "pr_auc": round(average_precision_score(y_val, y_proba), 4),
    "f1": round(f1_score(y_val, y_pred), 4),
    "accuracy": round(accuracy_score(y_val, y_pred), 4),
}

metrics_path = MODELS_DIR / "metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics saved: {metrics_path}")
for k, v in metrics.items():
    print(f"  {k:<15s} {v:.4f}")

Metrics saved: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/models/metrics.json
  auc_roc         0.7839
  pr_auc          0.2845
  f1              0.3408
  accuracy        0.8766


In [11]:
# Verify all artifacts
print("Deployment artifacts:")
print("=" * 60)

artifacts = [
    ("Model", MODELS_DIR / "best_xgb.json"),
    ("Preprocessor", MODELS_DIR / "preprocessor.pkl"),
    ("Feature columns", MODELS_DIR / "feature_columns.json"),
    ("Categorical columns", MODELS_DIR / "categorical_columns.json"),
    ("Metrics", MODELS_DIR / "metrics.json"),
]

for name, path in artifacts:
    exists = path.exists()
    size = path.stat().st_size / 1024 if exists else 0
    status = "OK" if exists else "MISSING"
    print(f"  [{status:>7s}] {name:<25s} {path.name:<35s} {size:>8.1f} KB")

Deployment artifacts:
  [     OK] Model                     best_xgb.json                         4481.0 KB
  [     OK] Preprocessor              preprocessor.pkl                         3.2 KB
  [     OK] Feature columns           feature_columns.json                     5.7 KB
  [     OK] Categorical columns       categorical_columns.json                 0.4 KB
  [     OK] Metrics                   metrics.json                             0.1 KB


## 4. Create FastAPI Serving Endpoint

The FastAPI application is defined in `serving/api.py`. It provides:

- `POST /predict` — Score a single loan application
- `GET /health` — Health check
- `GET /model-info` — Model metadata

Start with:
```bash
uvicorn serving.api:app --reload
```

In [12]:
# Display the serving API source
api_path = PROJECT_ROOT / "serving" / "api.py"
print(f"FastAPI endpoint: {api_path}")
print("=" * 70)
print(api_path.read_text())

FastAPI endpoint: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/serving/api.py
"""
FastAPI serving endpoint for Credit Risk prediction model.

Usage:
    uvicorn serving.api:app --reload

Expects model artifacts in the `models/` directory:
    - best_xgb.json          (XGBoost model)
    - preprocessor.pkl        (fitted OrdinalEncoder)
    - feature_columns.json    (ordered feature column list)
    - categorical_columns.json (categorical column names)
"""

from contextlib import asynccontextmanager
from pathlib import Path
from typing import Optional
import json
import pickle

import xgboost as xgb
import numpy as np
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

# ---------------------------------------------------------------------------
# Paths (relative to project root — works when launched from project root)
# ---------------------------------------------------------------------------
PROJECT_ROOT =

## 5. Create Streamlit Dashboard

The Streamlit dashboard is defined in `app/streamlit_dashboard.py`. It includes:

1. **Model Overview** — metrics & feature importance
2. **Make Prediction** — interactive form + SHAP explanation
3. **Data Explorer** — browse training data & distributions

Start with:
```bash
streamlit run app/streamlit_dashboard.py
```

In [13]:
# Display the Streamlit dashboard source
dashboard_path = PROJECT_ROOT / "app" / "streamlit_dashboard.py"
print(f"Streamlit dashboard: {dashboard_path}")
print("=" * 70)
print(dashboard_path.read_text())

Streamlit dashboard: /home/felipefrl/Documents/Coding/JupyterNotebooks/credit-risk-mlplatform/app/streamlit_dashboard.py
"""
Credit Risk Assessment Dashboard — Streamlit application.

Usage:
    streamlit run app/streamlit_dashboard.py

Expects model artifacts in the `models/` directory and gold layer data
in `data/features/`.
"""

import json
import pickle
from pathlib import Path

import xgboost as xgb
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).resolve().parent.parent
MODELS_DIR = PROJECT_ROOT / "models"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
FIGURES_DIR = PROJECT_ROOT / "figures"

# ---------------------------------------------------------------------------
# Page config
# ----------------------------

## 6. Test Serving Endpoint

In [14]:
# Start FastAPI server in a subprocess (non-blocking)
import os

server_process = subprocess.Popen(
    ["uvicorn", "serving.api:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env={**os.environ},
)

# Wait for server to start
print("Starting FastAPI server...")
time.sleep(5)
print(f"Server PID: {server_process.pid}")
print(f"Server running: {server_process.poll() is None}")

Starting FastAPI server...
Server PID: 95600
Server running: True


In [15]:
# Test health endpoint
import httpx

BASE_URL = "http://127.0.0.1:8000"

try:
    response = httpx.get(f"{BASE_URL}/health", timeout=10)
    print(f"GET /health")
    print(f"  Status: {response.status_code}")
    print(f"  Body:   {response.json()}")
except Exception as e:
    print(f"Health check failed: {e}")
    print("Server may not have started — check that all model artifacts exist.")

GET /health
  Status: 200
  Body:   {'status': 'healthy'}


In [16]:
# Test model-info endpoint
try:
    response = httpx.get(f"{BASE_URL}/model-info", timeout=10)
    print(f"GET /model-info")
    print(f"  Status: {response.status_code}")
    info = response.json()
    print(f"  Model type:        {info['model_type']}")
    print(f"  Number of features: {info['n_features']}")
    print(f"  Categorical cols:   {len(info['categorical_columns'])}")
except Exception as e:
    print(f"Model info request failed: {e}")

GET /model-info
  Status: 200
  Model type:        XGBoost
  Number of features: 225
  Categorical cols:   16


In [17]:
# Test prediction endpoint with a sample application
sample_application = {
    "AMT_INCOME_TOTAL": 202500.0,
    "AMT_CREDIT": 406597.5,
    "AMT_ANNUITY": 24700.5,
    "AMT_GOODS_PRICE": 351000.0,
    "AGE_YEARS": 35.0,
    "NAME_CONTRACT_TYPE": "Cash loans",
    "CODE_GENDER": "M",
    "FLAG_OWN_CAR": "Y",
    "FLAG_OWN_REALTY": "Y",
    "NAME_EDUCATION_TYPE": "Higher education",
    "NAME_FAMILY_STATUS": "Married",
    "NAME_INCOME_TYPE": "Working",
    "CNT_CHILDREN": 0,
    "CNT_FAM_MEMBERS": 2,
    "DAYS_EMPLOYED": -3000.0,
    "EXT_SOURCE_1": 0.5,
    "EXT_SOURCE_2": 0.6,
    "EXT_SOURCE_3": 0.5,
}

response = httpx.post(f"{BASE_URL}/predict", json=sample_application, timeout=60)
print(f"POST /predict — Status: {response.status_code}")
result = response.json()
if response.status_code == 200:
    print(f"  Probability:  {result['probability']:.4f}")
    print(f"  Prediction:   {result['prediction']} ({'DEFAULT' if result['prediction'] == 1 else 'NO DEFAULT'})")
    print(f"  Risk Level:   {result['risk_level']}")
else:
    print(f"  Error detail: {result}")

POST /predict — Status: 200
  Probability:  0.1505
  Prediction:   0 (NO DEFAULT)
  Risk Level:   Medium


In [18]:
# Check server is still alive, then test with a higher-risk profile
print(f"Server process alive: {server_process.poll() is None}")

# Re-check health
health = httpx.get(f"{BASE_URL}/health", timeout=10)
print(f"Health check: {health.status_code} {health.json()}")

high_risk_application = {
    "AMT_INCOME_TOTAL": 67500.0,
    "AMT_CREDIT": 900000.0,
    "AMT_ANNUITY": 45000.0,
    "AMT_GOODS_PRICE": 900000.0,
    "AGE_YEARS": 22.0,
    "NAME_CONTRACT_TYPE": "Cash loans",
    "CODE_GENDER": "M",
    "FLAG_OWN_CAR": "N",
    "FLAG_OWN_REALTY": "N",
    "NAME_EDUCATION_TYPE": "Lower secondary",
    "NAME_FAMILY_STATUS": "Single / not married",
    "NAME_INCOME_TYPE": "Working",
    "CNT_CHILDREN": 0,
    "CNT_FAM_MEMBERS": 1,
    "DAYS_EMPLOYED": -200.0,
    "EXT_SOURCE_1": 0.1,
    "EXT_SOURCE_2": 0.15,
    "EXT_SOURCE_3": 0.1,
}

response = httpx.post(f"{BASE_URL}/predict", json=high_risk_application, timeout=60)
print(f"POST /predict (high-risk) — Status: {response.status_code}")
result = response.json()
if response.status_code == 200:
    print(f"  Probability:  {result['probability']:.4f}")
    print(f"  Prediction:   {result['prediction']} ({'DEFAULT' if result['prediction'] == 1 else 'NO DEFAULT'})")
    print(f"  Risk Level:   {result['risk_level']}")
else:
    print(f"  Error detail: {result}")

Server process alive: True
Health check: 200 {'status': 'healthy'}
POST /predict (high-risk) — Status: 200
  Probability:  0.7422
  Prediction:   1 (DEFAULT)
  Risk Level:   Very High


In [19]:
# Stop the FastAPI server
server_process.terminate()
try:
    server_process.wait(timeout=10)
except Exception:
    server_process.kill()
    server_process.wait()
print(f"Server stopped (exit code: {server_process.returncode})")

Server stopped (exit code: -15)


## 7. Summary

### Deployment Architecture

```
data/features/              models/                     serving/         app/
  train_features.parquet      best_xgb.json               api.py          streamlit_dashboard.py
                              preprocessor.pkl
                              feature_columns.json
                              categorical_columns.json
                              metrics.json

MLFlow Registry: credit-risk-xgb (Production)
```

### How to Run

| Component | Command |
|-----------|--------|
| MLFlow UI | `mlflow ui --backend-store-uri sqlite:///mlruns/mlflow.db` |
| FastAPI   | `uvicorn serving.api:app --reload` |
| Streamlit | `streamlit run app/streamlit_dashboard.py` |

### What We Accomplished Across All 9 Notebooks

| Notebook | Task |
|----------|------|
| **01** | Data ingestion: CSV to Parquet (bronze -> silver) |
| **02** | Exploratory data analysis: distributions, correlations, target analysis |
| **03** | Feature engineering: aggregations, domain features (silver -> gold) |
| **04** | Feature store setup with Feast |
| **05** | Baseline models (LogReg, RF) + MLFlow experiment tracking |
| **06** | Model selection & comparison (LightGBM, XGBoost, CatBoost) + SHAP |
| **07** | Hyperparameter tuning with Optuna + MLFlow |
| **08** | Evaluation & insights: SHAP, error analysis, business metrics |
| **09** | Model registry, FastAPI serving, Streamlit dashboard |

**End-to-end ML platform complete** — from raw data to a production-ready API and interactive dashboard.